# Prime Numbers Lab — Notebook 13: Gap Distribution Residual Structure

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Notebook purpose:** measure how normalized prime-gap distributions deviate from the exponential baseline.

**Core frame:**  
Constraint → structure remains under constraint; drift marks invalid assignments; structure may remain recoverable from partial observation.

Notebook 12 showed that normalized prime gaps:

\[
z = \frac{p_{n+1}-p_n}{\log p_n}
\]

approximately follow an \(\mathrm{Exp}(1)\) distribution.

Notebook 13 studies the residual:

\[
\Delta(z,x)=f_{\mathrm{emp}}(z,x)-e^{-z}
\]

so the analysis moves from “fits exponential” to “where and how does it deviate?”

## 0. Setup

This notebook follows the established `prime-numbers-lab` template:

1. define one constraint  
2. generate one dataset  
3. measure what remains under constraint  
4. visualize drift / retention / recoverability  
5. export figures, data, notes, and TeX  
6. package results into a root-level export zip

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

NOTEBOOK_ID = "13_gap_distribution_residual_structure"
NOTEBOOK_TITLE = "Gap Distribution Residual Structure"
REPO_NAME = "prime-numbers-lab"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]

OUT = Path(NOTEBOOK_ID)
FIG_DIR = OUT / "figures"
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
TEX_DIR = OUT / "tex"

for d in [FIG_DIR, DATA_DIR, DOCS_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUT.resolve()}")

## 1. Premise

Notebook 12 showed that normalized prime gaps broadly align with the exponential baseline.

Notebook 13 measures the residual structure:

\[
\Delta(z,x)=f_{\mathrm{emp}}(z,x)-e^{-z}
\]

**Short vocabulary:**

- **Remains under constraint / persists:** exponential envelope remains visible after normalization.
- **Drift:** measurable residual deviation from \(\mathrm{Exp}(1)\).
- **Recoverability:** whether residual magnitude decreases across scale and reveals stable correction structure.

Core question:

> Does residual structure shrink with scale, or does a persistent correction remain?

## 2. Constraint definition

The baseline from Notebook 12 is:

\[
z=\frac{g_n}{\log p_n}
\]

where:

\[
g_n=p_{n+1}-p_n
\]

The exponential reference distribution is:

\[
f_0(z)=e^{-z}, \quad z\ge 0
\]

The residual is:

\[
\Delta(z,x)=f_{\mathrm{emp}}(z,x)-f_0(z)
\]

The main measurement is:

\[
\|\Delta(\cdot,x)\|_1
=
\int |\Delta(z,x)|\,dz
\]

estimated from histogram bins.

In [ ]:
N_MAX = 2_000_000
RANDOM_SEED = 9423

Z_MAX = 6.0
BIN_COUNT = 90

WINDOW_COUNT = 18
MIN_WINDOW_GAPS = 200

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "Z_MAX": Z_MAX,
    "BIN_COUNT": BIN_COUNT,
    "WINDOW_COUNT": WINDOW_COUNT,
    "MIN_WINDOW_GAPS": MIN_WINDOW_GAPS,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
}

params

## 3. Data generation

Generate primes up to \(N_{\max}\), then compute gaps and normalized gaps.

In [ ]:
def generate_primes(n_max: int) -> np.ndarray:
    if n_max < 2:
        return np.array([], dtype=int)

    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False

    for i in range(2, int(math.sqrt(n_max)) + 1):
        if sieve[i]:
            sieve[i*i:n_max+1:i] = False

    return np.nonzero(sieve)[0].astype(int)

primes = generate_primes(N_MAX)
gaps = np.diff(primes)
anchors = primes[:-1]
normalized_gaps = gaps / np.log(anchors)

summary = {
    "n_max": int(N_MAX),
    "prime_count": int(len(primes)),
    "gap_count": int(len(gaps)),
    "first_primes": primes[:10].tolist(),
    "last_primes": primes[-10:].tolist(),
    "mean_gap": float(np.mean(gaps)),
    "mean_normalized_gap": float(np.mean(normalized_gaps)),
    "std_normalized_gap": float(np.std(normalized_gaps)),
}

summary

## 4. Window construction

Use logarithmically spaced windows to measure scale-dependent residuals.

Each window collects gaps whose left prime \(p_n\) lies inside the window.

In [ ]:
raw_edges = np.unique(np.logspace(np.log10(100), np.log10(N_MAX), WINDOW_COUNT + 1).astype(int))
raw_edges[0] = 2
raw_edges[-1] = N_MAX

window_rows = []
window_gap_arrays = []

for idx, (left, right) in enumerate(zip(raw_edges[:-1], raw_edges[1:]), start=1):
    mask = (anchors >= left) & (anchors < right)
    z = normalized_gaps[mask]
    g = gaps[mask]
    x = anchors[mask]

    if len(z) < MIN_WINDOW_GAPS:
        continue

    row = {
        "window_index": idx,
        "left": int(left),
        "right": int(right),
        "midpoint": float(math.sqrt(left * right)),
        "gap_count": int(len(z)),
        "mean_normalized_gap": float(np.mean(z)),
        "std_normalized_gap": float(np.std(z)),
        "median_normalized_gap": float(np.median(z)),
        "q75_normalized_gap": float(np.quantile(z, 0.75)),
        "q90_normalized_gap": float(np.quantile(z, 0.90)),
        "q95_normalized_gap": float(np.quantile(z, 0.95)),
    }

    window_rows.append(row)
    window_gap_arrays.append({
        "window_index": idx,
        "left": int(left),
        "right": int(right),
        "midpoint": float(math.sqrt(left * right)),
        "z": z,
        "gaps": g,
        "anchors": x,
    })

windows_df = pd.DataFrame(window_rows)

windows_df.head(), windows_df.tail()

## 5. Measurement

Estimate empirical PDFs and compare each to the exponential baseline.

Outputs:

- residual curves
- residual \(L_1\), \(L_2\), and signed mean
- positive / negative residual mass
- Jensen–Shannon divergence
- residual tail bias

In [ ]:
def safe_normalize_pdf(pdf: np.ndarray, bin_width: float) -> np.ndarray:
    total = float(np.sum(pdf) * bin_width)
    if total <= 0:
        return pdf
    return pdf / total

def js_divergence_discrete(p_density: np.ndarray, q_density: np.ndarray, bin_width: float) -> float:
    p = np.maximum(p_density * bin_width, 1e-15)
    q = np.maximum(q_density * bin_width, 1e-15)

    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))

    return float(0.5 * (kl_pm + kl_qm))

bins = np.linspace(0, Z_MAX, BIN_COUNT + 1)
centers = 0.5 * (bins[:-1] + bins[1:])
bin_width = float(bins[1] - bins[0])

exp_pdf = np.exp(-centers)
exp_pdf = safe_normalize_pdf(exp_pdf, bin_width)

residual_rows = []
residual_grid_rows = []

for item in window_gap_arrays:
    idx = item["window_index"]
    z = item["z"]

    hist, _ = np.histogram(z, bins=bins, density=True)
    hist = safe_normalize_pdf(hist, bin_width)

    delta = hist - exp_pdf

    l1 = float(np.sum(np.abs(delta)) * bin_width)
    l2 = float(np.sqrt(np.sum(delta**2) * bin_width))
    signed_mean = float(np.sum(delta) * bin_width)

    positive_mass = float(np.sum(np.maximum(delta, 0)) * bin_width)
    negative_mass = float(np.sum(np.minimum(delta, 0)) * bin_width)

    js = js_divergence_discrete(hist, exp_pdf, bin_width)

    tail_mask = centers >= 3.0
    tail_bias = float(np.sum(delta[tail_mask]) * bin_width)

    residual_rows.append({
        "window_index": idx,
        "left": item["left"],
        "right": item["right"],
        "midpoint": item["midpoint"],
        "gap_count": int(len(z)),
        "residual_l1": l1,
        "residual_l2": l2,
        "residual_signed_mean": signed_mean,
        "positive_residual_mass": positive_mass,
        "negative_residual_mass": negative_mass,
        "js_divergence": js,
        "tail_bias_z_ge_3": tail_bias,
    })

    for c, h, e, d in zip(centers, hist, exp_pdf, delta):
        residual_grid_rows.append({
            "window_index": idx,
            "midpoint": item["midpoint"],
            "z_center": float(c),
            "empirical_pdf": float(h),
            "exp_pdf": float(e),
            "delta": float(d),
        })

residual_metrics_df = pd.DataFrame(residual_rows)
residual_grid_df = pd.DataFrame(residual_grid_rows)

measurement = {
    "mean_residual_l1": float(residual_metrics_df["residual_l1"].mean()),
    "final_residual_l1": float(residual_metrics_df["residual_l1"].iloc[-1]),
    "mean_residual_l2": float(residual_metrics_df["residual_l2"].mean()),
    "final_residual_l2": float(residual_metrics_df["residual_l2"].iloc[-1]),
    "mean_js_divergence": float(residual_metrics_df["js_divergence"].mean()),
    "final_js_divergence": float(residual_metrics_df["js_divergence"].iloc[-1]),
    "mean_tail_bias_z_ge_3": float(residual_metrics_df["tail_bias_z_ge_3"].mean()),
    "final_tail_bias_z_ge_3": float(residual_metrics_df["tail_bias_z_ge_3"].iloc[-1]),
    "window_count_used": int(len(residual_metrics_df)),
}

measurement

## 6. CGCS score

The score measures residual agreement with the exponential baseline.

Working definition:

\[
CGCS_{\Delta}
=
\frac{1}{1+\overline{\|\Delta\|_1}+\overline{JS}}
\]

Higher score indicates stronger residual agreement with the exponential baseline.

In [ ]:
cgcs_score = 1.0 / (
    1.0
    + measurement["mean_residual_l1"]
    + measurement["mean_js_divergence"]
)

cgcs = {
    "score": float(cgcs_score),
    "definition": "1 / (1 + mean residual L1 + mean JS divergence)",
    "interpretation": "Closer to 1 indicates smaller residual deviation from the exponential baseline.",
}

cgcs

## 7. Visualization

Notebook 13 produces residual-focused figures:

1. residual curves  
2. residual heatmap  
3. residual norm vs scale  
4. JS divergence vs scale  
5. positive / negative residual mass  
6. tail bias  
7. empirical vs exponential PDF in final window  
8. residual sign map

### Figure 1 — residual curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for idx in residual_metrics_df["window_index"].tolist():
    sub = residual_grid_df[residual_grid_df["window_index"] == idx]
    ax.plot(sub["z_center"], sub["delta"], alpha=0.35)

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Residual curves: empirical PDF minus Exp(1)")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("Delta(z, x)")
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_residual_curves.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

### Figure 2 — residual heatmap

In [ ]:
pivot_delta = residual_grid_df.pivot(index="window_index", columns="z_center", values="delta")

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    pivot_delta.values,
    aspect="auto",
    origin="lower",
    interpolation="nearest",
    extent=[centers.min(), centers.max(), 0, len(pivot_delta.index)],
)
ax.axvline(1.0, linestyle="--", linewidth=1)
ax.axvline(3.0, linestyle="--", linewidth=1)
ax.set_title("Residual heatmap across scale")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("window index")
fig.colorbar(im, ax=ax, label="Delta(z, x)")

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_residual_heatmap.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

### Figure 3 — residual norm vs scale

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(residual_metrics_df["midpoint"], residual_metrics_df["residual_l1"], marker="o", label="L1 residual")
ax.plot(residual_metrics_df["midpoint"], residual_metrics_df["residual_l2"], marker="o", label="L2 residual")
ax.set_xscale("log")
ax.set_title("Residual norm vs scale")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("residual norm")
ax.legend()
ax.grid(True, alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_residual_norm_vs_scale.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

### Figure 4 — JS divergence vs scale

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(residual_metrics_df["midpoint"], residual_metrics_df["js_divergence"], marker="o")
ax.set_xscale("log")
ax.set_title("JS divergence to Exp(1) vs scale")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("JS divergence")
ax.grid(True, alpha=0.3)

fig4_path = FIG_DIR / f"{NOTEBOOK_NUM}_js_divergence_vs_scale.png"
fig.savefig(fig4_path, dpi=180, bbox_inches="tight")
plt.show()

fig4_path

### Figure 5 — positive and negative residual mass

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(residual_metrics_df["midpoint"], residual_metrics_df["positive_residual_mass"], marker="o", label="positive residual mass")
ax.plot(residual_metrics_df["midpoint"], -residual_metrics_df["negative_residual_mass"], marker="o", label="negative residual mass magnitude")
ax.set_xscale("log")
ax.set_title("Positive / negative residual mass")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("mass")
ax.legend()
ax.grid(True, alpha=0.3)

fig5_path = FIG_DIR / f"{NOTEBOOK_NUM}_positive_negative_residual_mass.png"
fig.savefig(fig5_path, dpi=180, bbox_inches="tight")
plt.show()

fig5_path

### Figure 6 — tail bias

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(residual_metrics_df["midpoint"], residual_metrics_df["tail_bias_z_ge_3"], marker="o")
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_title("Tail residual bias for z >= 3")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("tail bias")
ax.grid(True, alpha=0.3)

fig6_path = FIG_DIR / f"{NOTEBOOK_NUM}_tail_bias_z_ge_3.png"
fig.savefig(fig6_path, dpi=180, bbox_inches="tight")
plt.show()

fig6_path

### Figure 7 — final window PDF comparison

In [ ]:
final_idx = int(residual_metrics_df["window_index"].iloc[-1])
final_sub = residual_grid_df[residual_grid_df["window_index"] == final_idx].copy()

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(final_sub["z_center"], final_sub["empirical_pdf"], marker="o", markersize=3, label="empirical PDF")
ax.plot(final_sub["z_center"], final_sub["exp_pdf"], linewidth=2, label="Exp(1)")
ax.set_title("Final window PDF vs Exp(1)")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("density")
ax.legend()
ax.grid(True, alpha=0.3)

fig7_path = FIG_DIR / f"{NOTEBOOK_NUM}_final_window_pdf_vs_exp1.png"
fig.savefig(fig7_path, dpi=180, bbox_inches="tight")
plt.show()

fig7_path

### Figure 8 — residual sign map

In [ ]:
sign_map = np.sign(pivot_delta.values)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    sign_map,
    aspect="auto",
    origin="lower",
    interpolation="nearest",
    extent=[centers.min(), centers.max(), 0, len(pivot_delta.index)],
)
ax.set_title("Residual sign map")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("window index")
fig.colorbar(im, ax=ax, label="sign of Delta")

fig8_path = FIG_DIR / f"{NOTEBOOK_NUM}_residual_sign_map.png"
fig.savefig(fig8_path, dpi=180, bbox_inches="tight")
plt.show()

fig8_path

## 8. Interpretation

Use a short, consistent structure:

1. **What remains under constraint?**  
2. **What drifts?**  
3. **What appears recoverable?**  
4. **What should not be overclaimed?**

In [ ]:
interpretation = f'''
# {NOTEBOOK_TITLE}

## Constraint result

This notebook measures residual structure in normalized prime gaps after comparison to the exponential baseline.

The residual is:

$$
\\Delta(z,x)=f_{{emp}}(z,x)-e^{{-z}}.
$$

## Remains under constraint

The exponential envelope remains visible after normalization by $\\log p_n$.

The final-window divergence and residual norms remain finite and measurable:

- final residual L1 = {measurement["final_residual_l1"]:.6f}
- final residual L2 = {measurement["final_residual_l2"]:.6f}
- final JS divergence = {measurement["final_js_divergence"]:.6f}

## Drift

Drift is measured as structured residual deviation from $\\mathrm{{Exp}}(1)$.

The residual is not treated as random noise. Positive and negative residual regions show where empirical density exceeds or falls below the exponential baseline.

## Tail bias

The tail-bias metric measures residual mass for $z \\ge 3$:

- mean tail bias = {measurement["mean_tail_bias_z_ge_3"]:.6f}
- final tail bias = {measurement["final_tail_bias_z_ge_3"]:.6f}

## Recoverability

Residual structure is recoverable as a windowed diagnostic:

- residual curves
- residual heatmap
- residual norm vs scale
- JS divergence vs scale
- tail bias

## CGCS score

The residual agreement score is:

$$
CGCS_\\Delta = \\frac{{1}}{{1+\\overline{{\\|\\Delta\\|_1}}+\\overline{{JS}}}}.
$$

Measured score:

$$
CGCS_\\Delta = {cgcs_score:.6f}.
$$

## Caution

This notebook does not prove a new theorem about prime gaps.

It measures finite residual structure relative to the exponential heuristic.
'''.strip()

print(interpretation)

## 9. Export data, notes, figures index, math, and TeX

This block writes reusable artifacts:

- CSV summary
- JSON metadata
- Markdown interpretation with embedded figure links
- Markdown design notes
- TeX results snippet
- standalone TeX math notes

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
windows_path = DATA_DIR / f"{NOTEBOOK_NUM}_windows.csv"
residual_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_residual_metrics.csv"
residual_grid_path = DATA_DIR / f"{NOTEBOOK_NUM}_residual_grid.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_notes_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
windows_df.to_csv(windows_path, index=False)
residual_metrics_df.to_csv(residual_metrics_path, index=False)
residual_grid_df.to_csv(residual_grid_path, index=False)

figure_paths = [
    fig1_path,
    fig2_path,
    fig3_path,
    fig4_path,
    fig5_path,
    fig6_path,
    fig7_path,
    fig8_path,
]

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "windows": str(windows_path),
        "residual_metrics": str(residual_metrics_path),
        "residual_grid": str(residual_grid_path),
    },
    "docs": {
        "interpretation": str(interpretation_md_path),
        "design_notes": str(design_notes_md_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

figures_md = "\n\n## Figures\n\n"
for i, fig in enumerate(figure_paths, start=1):
    figures_md += f"### Figure {i} — {fig.stem.replace('_', ' ').title()}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

interpretation_md_path.write_text(interpretation + figures_md, encoding="utf-8")

design_notes = f'''
# Design Notes — {NOTEBOOK_TITLE}

## Notebook role

Notebook 13 follows Notebook 12 by moving from exponential distribution comparison to residual structure.

## Constraint

The constraint is the exponential baseline for normalized prime gaps:

$$
z = \\frac{{p_{{n+1}}-p_n}}{{\\log p_n}}, \\quad f_0(z)=e^{{-z}}.
$$

## Measurement

The measured object is:

$$
\\Delta(z,x)=f_{{emp}}(z,x)-e^{{-z}}.
$$

Metrics include residual L1, residual L2, JS divergence, signed residual mass, and tail bias.

## CGCS score

$$
CGCS_\\Delta = \\frac{{1}}{{1+\\overline{{\\|\\Delta\\|_1}}+\\overline{{JS}}}}.
$$

## Handoff

Notebook 14 should test whether residual magnitude follows a scaling law such as $1/\\log x$ or $1/(\\log x)^\\alpha$.
'''.strip()

design_notes_md_path.write_text(design_notes + "\n", encoding="utf-8")

summary_tex = rf'''
\section*{{{NOTEBOOK_TITLE}}}

This notebook measures residual structure after comparing normalized prime gaps to the exponential baseline.

\[
z = \frac{{p_{{n+1}}-p_n}}{{\log p_n}}
\]

\[
\Delta(z,x)=f_{{emp}}(z,x)-e^{{-z}}
\]

\begin{{itemize}}
  \item Windows used: {measurement["window_count_used"]}
  \item Mean residual $L_1$: {measurement["mean_residual_l1"]:.6f}
  \item Final residual $L_1$: {measurement["final_residual_l1"]:.6f}
  \item Mean JS divergence: {measurement["mean_js_divergence"]:.6f}
  \item Final JS divergence: {measurement["final_js_divergence"]:.6f}
  \item Mean tail bias for $z\ge3$: {measurement["mean_tail_bias_z_ge_3"]:.6f}
  \item CGCS residual score: {cgcs_score:.6f}
\end{{itemize}}

The residual is treated as a measurable correction to the exponential heuristic, not as noise to ignore.
'''.strip()

summary_tex_path.write_text(summary_tex + "\n", encoding="utf-8")

math_tex = rf'''
\documentclass{{article}}
\usepackage{{amsmath}}
\usepackage{{amssymb}}
\usepackage[margin=1in]{{geometry}}

\begin{{document}}

\section*{{Math Notes: {NOTEBOOK_TITLE}}}

\subsection*{{Normalized gap}}

\[
z_n = \frac{{p_{{n+1}}-p_n}}{{\log p_n}}
\]

\subsection*{{Exponential baseline}}

\[
f_0(z)=e^{{-z}}, \quad z\ge 0
\]

\subsection*{{Residual}}

\[
\Delta(z,x)=f_{{emp}}(z,x)-f_0(z)
\]

\subsection*{{Residual norms}}

\[
\|\Delta\|_1 = \int |\Delta(z,x)|\,dz
\]

\[
\|\Delta\|_2 = \left(\int \Delta(z,x)^2\,dz\right)^{{1/2}}
\]

\subsection*{{Jensen--Shannon divergence}}

\[
JS(P,Q)=\frac12 KL(P\|M)+\frac12 KL(Q\|M)
\]

where:

\[
M=\frac12(P+Q)
\]

\subsection*{{Residual CGCS score}}

\[
CGCS_\Delta =
\frac{{1}}{{1+\overline{{\|\Delta\|_1}}+\overline{{JS}}}}
\]

\subsection*{{Interpretation}}

Structure remains under constraint when normalized gaps retain an exponential envelope while residual deviations become measurable, bounded, and comparable across scale.

\end{{document}}
'''.strip()

math_tex_path.write_text(math_tex + "\n", encoding="utf-8")

summary_path, windows_path, residual_metrics_path, residual_grid_path, metadata_path, interpretation_md_path, design_notes_md_path, summary_tex_path, math_tex_path

## 10. Optional results bundle

This creates a root-level export zip containing figures, data, docs, and TeX outputs.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 11. Next notebook handoff

Next:

> Notebook 14 should test whether residual magnitude follows a scaling law such as \(1/\log x\) or \(1/(\log x)^\alpha\).

In [ ]:
next_step = "Notebook 14: residual scaling law."
print(next_step)